# Pelatihan M1 (ResNet50) dan M2 (ResNet50-GLCM)
Notebook ini melatih **baseline M1** dan **model usulan M2** sesuai Bab 4 laporan.

**Cara pakai:**
1. Upload **aksara-sunda-resnet50-glcm.zip** (cukup satu file zip, tidak perlu diekstrak) ke **My Drive**. Bisa juga folder hasil ekstraknya.
2. *Runtime → Change runtime type → **T4 GPU***.
3. *Runtime → **Run all***, lalu izinkan akses Google Drive saat diminta.

Zip diekstrak otomatis ke `My Drive/aksara-sunda-resnet50-glcm/`, dan semua hasil (model, grafik) tersimpan di sana.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, zipfile

DRIVE = '/content/drive/MyDrive'
NAMA = ['aksara-sunda-resnet50-glcm', 'Preprocessing_ResNet50_GLCM']   # nama folder/zip yang dikenali

def cari_proyek():
    return next((f'{DRIVE}/{n}' for n in NAMA if os.path.exists(f'{DRIVE}/{n}/config.py')), None)

PROJECT_DIR = cari_proyek()
if PROJECT_DIR is None:
    zips = [f'{DRIVE}/{n}.zip' for n in NAMA if os.path.exists(f'{DRIVE}/{n}.zip')]
    assert zips, 'Upload aksara-sunda-resnet50-glcm.zip ke My Drive terlebih dahulu.'
    print('Mengekstrak', zips[0], '...')
    zipfile.ZipFile(zips[0]).extractall(DRIVE)
    PROJECT_DIR = cari_proyek()

%cd "$PROJECT_DIR"
n_kelas = len([d for d in os.listdir('dataset_gabung_fix') if os.path.isdir(f'dataset_gabung_fix/{d}')])
print('Folder proyek:', PROJECT_DIR)
print('Folder kelas:', n_kelas, '| eda_output ada:', os.path.exists('eda_output/split_train.csv'))

In [ ]:
!pip install -q scikit-image
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 1. Preprocessing
Dilewati otomatis bila `preprocessing_output/` sudah ada (cukup dijalankan sekali untuk semua model).

In [ ]:
import os
if not os.path.exists('preprocessing_output/preprocessed_224.npz'):
    !python run_preprocessing.py
else:
    print('Preprocessing sudah ada, dilewati.')
print(open('preprocessing_output/ringkasan_preprocessing.txt').read())

## 2. Pelatihan baseline M1 (ResNet50)
Tahap 1: backbone ResNet50 (bobot ImageNet) dibekukan, hanya kepala klasifikasi yang dilatih (lr 1e-3).
Tahap 2: 30 lapisan akhir dibuka untuk fine-tuning (lr 1e-5). Early stopping (patience 5) dan bobot terbaik berdasarkan `val_loss`.
Estimasi waktu di GPU T4: ±10–30 menit.

In [ ]:
!python train_model.py --model M1

## 3. Pelatihan model usulan M2 (ResNet50-GLCM)
Parameter sama dengan M1; bedanya hanya tambahan 32 fitur GLCM yang digabung sebelum lapisan klasifikasi.

In [ ]:
!python train_model.py --model M2

## 4. Hasil dan perbandingan M1 vs M2

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
keys = ['akurasi_uji','precision_macro','recall_macro','f1_macro','epoch_total','epoch_terbaik',
        'val_akurasi_pada_epoch_terbaik','waktu_latih_menit','waktu_inferensi_ms_per_citra','jumlah_parameter']
hasil = {m: json.load(open(f'hasil_{m}/ringkasan_hasil.json')) for m in ['M1','M2']}
tabel = pd.DataFrame({m: {k: r[k] for k in keys} for m, r in hasil.items()})
display(tabel)
tabel.to_csv('perbandingan_M1_M2.csv')
for m, r in hasil.items():
    print(f'\n=== {m} ({r["nama"]}) ===')
    print('Kelas paling sering tertukar (asli, prediksi, jumlah):', r['pasangan_kelas_tertukar_terbanyak'])
    display(Image(f'hasil_{m}/kurva_pelatihan.png'))
    display(Image(f'hasil_{m}/confusion_matrix.png'))

## 5. (Opsional) M3 MobileNetV2 dan M4 MobileNetV2-GLCM
Hapus tanda `#` lalu jalankan sel ini bila waktu GPU masih cukup.

In [ ]:
# !python train_model.py --model M3
# !python train_model.py --model M4